In [1]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import featureman.gen_data as man
from sklearn.cluster import SpectralClustering
import pickle
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
model_dict = torch.load("two_layer_modular_arithmetic_model.pth", map_location=device)
model = man.MultiLayerTransformer(p=113, d_model=128, n_layers=2).to(device)
model.load_state_dict(model_dict)

<All keys matched successfully>

In [3]:
torch.manual_seed(1337)
# generate combination of all inputs a and b range (113)
a = np.arange(113)
b = np.arange(113)
# generate inputs for the model
inputs = np.array([[a_i, 113, b_i, 114] for a_i in a for b_i in b])
inputs = torch.tensor(inputs).to(device)  # Add batch dimension
print(inputs.shape)
logits, activations = model(inputs, return_activations=True)

torch.Size([12769, 4])


In [9]:
from sklearn.decomposition import PCA
import plotly.graph_objects as go
import numpy as np

pca = PCA()
output_pca = pca.fit_transform(activations[1][:,-1,:].detach().cpu().numpy())

# Create single 3D plot
fig = go.Figure()

for a in range(3):
    fig.add_trace(
        go.Scatter3d(
            x=output_pca[:, 0][a*113:a*113+113],
            y=output_pca[:, 1][a*113:a*113+113],
            z=output_pca[:, 5][a*113:a*113+113],
            mode='markers',
            marker=dict(
                size=2,
                colorscale='viridis',
                opacity=0.3,
                symbol='x'
            ),
            name=f'Batch {a}'
        )
    )

# Update scene properties
fig.update_layout(
    scene=dict(
        xaxis_title="Principal Component 0",
        yaxis_title="Principal Component 1",
        zaxis_title="Principal Component 5"
    ),
    title="PCA Visualization of Reconstructions",
    width=800,
    height=700
)

fig.show()

In [11]:
from sklearn.decomposition import PCA
import plotly.graph_objects as go
import numpy as np


pca = PCA()
output_pca = pca.fit_transform(activations[0][:,-1,:].detach().cpu().numpy())

a_values = np.arange(113)
b_values = np.arange(113)

inputs = np.array([[a_i, 113, b_i, 114] for a_i in a_values for b_i in b_values])

targets = [
    50,
    51,
    52,
]

# Create single 3D plot
fig = go.Figure()

for i in targets:
    indices = np.where(inputs[:, 0] + inputs[:, 2] == i)[0]
    fig.add_trace(
        go.Scatter3d(
            x=output_pca[:, 0][indices],
            y=output_pca[:, 1][indices],
            z=output_pca[:, 5][indices],
            mode='markers',
            marker=dict(
                size=5,
                colorscale='viridis',
                opacity=0.6,
                symbol='x'
            ),
            name=f'Target {i}'
        )
    )

# Update scene properties
fig.update_layout(
    scene=dict(
        xaxis_title="Principal Component 0",
        yaxis_title="Principal Component 2",
        zaxis_title="Principal Component 5"
    ),
    title="PCA Visualization of Reconstructions",
    width=800,
    height=700
)

fig.show()